# 20 — Mini Data Wrangling Project

## Goal
To complete a small end-to-end cleaning and feature engineering workflow.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "raw" / "motionsense_raw.csv"
PROCESSED_DIR = DATA_DIR / "processed"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## Mini project workflow
In this notebook, we combine all the Data Cleaning & Feature Engineering skills learned so far:

1. Load raw data
2. Inspect the target
3. Fill missing sensor values
4. Remove duplicates
5. Create useful features
6. Export clean and ML-ready datasets

In [2]:
# Step 1: Load raw data.
df = pd.read_csv(RAW_PATH)

print("Raw dataset shape:", df.shape)
df.head()

Raw dataset shape: (1320, 11)


,row_id,subject,time_step,activity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,device
0,1,3,0,Walking,-0.190664,0.687583,1.129328,0.005386,-0.143240,0.014062,phone_A
1,2,24,1,Walking,0.010593,0.542629,0.998704,0.316734,0.085557,0.007263,phone_B
2,3,13,2,Walking,0.221782,0.375365,1.277480,0.114523,0.096630,-0.005492,phone_A
3,4,25,3,Walking,0.167558,0.386942,1.495453,0.203002,-0.047116,-0.038735,phone_A
4,5,11,4,Walking,0.330682,0.558338,1.472442,0.455581,-0.044706,-0.056347,phone_A


In [3]:
# Step 2: Inspect the target column.
print(df["activity"].value_counts())

activity
Walking               220
Walking Upstairs      220
Walking Downstairs    220
Sitting               220
Standing              220
Laying                220
Name: count, dtype: int64


In [4]:
# Step 3: Create a clean working copy.
clean_df = df.copy()

# Standardize text in the activity column.
clean_df["activity"] = clean_df["activity"].str.strip()

# Fill missing sensor values with each column mean.
sensor_columns = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]

for column in sensor_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")
    clean_df[column] = clean_df[column].fillna(clean_df[column].mean())

# Remove duplicate rows and reset row numbers.
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("Clean dataset shape:", clean_df.shape)
clean_df.head()

Clean dataset shape: (1320, 11)


,row_id,subject,time_step,activity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,device
0,1,3,0,Walking,-0.190664,0.687583,1.129328,0.005386,-0.143240,0.014062,phone_A
1,2,24,1,Walking,0.010593,0.542629,0.998704,0.316734,0.085557,0.007263,phone_B
2,3,13,2,Walking,0.221782,0.375365,1.277480,0.114523,0.096630,-0.005492,phone_A
3,4,25,3,Walking,0.167558,0.386942,1.495453,0.203002,-0.047116,-0.038735,phone_A
4,5,11,4,Walking,0.330682,0.558338,1.472442,0.455581,-0.044706,-0.056347,phone_A


In [5]:
# Step 4: Create magnitude features.
clean_df["acc_magnitude"] = np.sqrt(
    clean_df["acc_x"]**2 + clean_df["acc_y"]**2 + clean_df["acc_z"]**2
)

clean_df["gyro_magnitude"] = np.sqrt(
    clean_df["gyro_x"]**2 + clean_df["gyro_y"]**2 + clean_df["gyro_z"]**2
)

clean_df[["activity", "acc_magnitude", "gyro_magnitude"]].head()

,activity,acc_magnitude,gyro_magnitude
0,Walking,1.335853,0.144029
1,Walking,1.136647,0.328166
2,Walking,1.349830,0.149943
3,Walking,1.553763,0.211967
4,Walking,1.609092,0.461224


In [6]:
# Step 5: Create an ML-ready copy with an activity code.
ml_ready_df = clean_df.copy()

activity_map = {
    activity: code
    for code, activity in enumerate(sorted(ml_ready_df["activity"].unique()))
}

ml_ready_df["activity_code"] = ml_ready_df["activity"].map(activity_map)

print(activity_map)
ml_ready_df.head()

{'Laying': 0, 'Sitting': 1, 'Standing': 2, 'Walking': 3, 'Walking Downstairs': 4, 'Walking Upstairs': 5}


,row_id,subject,time_step,activity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,device,acc_magnitude,gyro_magnitude,activity_code
0,1,3,0,Walking,-0.190664,0.687583,1.129328,0.005386,-0.143240,0.014062,phone_A,1.335853,0.144029,3
1,2,24,1,Walking,0.010593,0.542629,0.998704,0.316734,0.085557,0.007263,phone_B,1.136647,0.328166,3
2,3,13,2,Walking,0.221782,0.375365,1.277480,0.114523,0.096630,-0.005492,phone_A,1.349830,0.149943,3
3,4,25,3,Walking,0.167558,0.386942,1.495453,0.203002,-0.047116,-0.038735,phone_A,1.553763,0.211967,3
4,5,11,4,Walking,0.330682,0.558338,1.472442,0.455581,-0.044706,-0.056347,phone_A,1.609092,0.461224,3


In [7]:
# Step 6: Export processed datasets.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_path = PROCESSED_DIR / "motionsense_clean.csv"
ml_ready_path = PROCESSED_DIR / "motionsense_ml_ready.csv"

clean_df.to_csv(clean_path, index=False)
ml_ready_df.to_csv(ml_ready_path, index=False)

print("Saved clean dataset to:", clean_path)
print("Saved ML-ready dataset to:", ml_ready_path)

Saved clean dataset to: ../data/processed/motionsense_clean.csv
Saved ML-ready dataset to: ../data/processed/motionsense_ml_ready.csv


## Summary
Raw sensor data has been converted into a cleaner, feature-rich dataset that is ready for statistics, visualization, and modeling.